DH演習2026 · 教材 第37版 · 2026-r37 · 2026-10-15

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第37版
教材ID: 2026-r37
教材日付: 2026-10-15
活動: source_trial_illustrated-class3
環境: Colab / Jupyter
ソース: 4aca4833f48594508442384d225b3d71866ef1f1
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第3回：領域を座標で表し、分類する

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：[画本虫撰](https://codh.rois.ac.jp/pmjt/book/200014778/)（日本古典籍データセット、国文研所蔵、CODH配信、[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)）

## 1. 画像を取得する

`region`は切り出す範囲（左,上,幅,高さ）で、原画像のピクセルで指定します。

In [ ]:
#@title 操作：道具を読み込む
from PIL import Image, ImageDraw

In [ ]:
koma = 5
region = "842,365,3875,3061"
url = f"https://codh.rois.ac.jp/pmjt/iiif/200014778/200014778_{koma:05d}.tif/{region}/2400,/0/default.jpg"
!curl -fso page.jpg {url}
Image.open("page.jpg")

## 2. OCRの枠を下書きにする

[NDL古典籍OCR-Lite](https://github.com/ndl-lab/ndlkotenocr-lite/tree/ede4283845cdc0ba2bda8b7ebfc3dc80b33c92c8)が検出した行ごとに、種類・左・上・幅・高さ・文字列の先頭を表示します。座標は`page.jpg`の左上からのピクセル数で、`region`の原画像の座標とは基準が違います。初回だけ、準備に約20秒かかります。

In [ ]:
#@title 操作：OCRの実行環境を準備する
ocr = "https://github.com/ndl-lab/ndlkotenocr-lite/archive/ede4283845cdc0ba2bda8b7ebfc3dc80b33c92c8.tar.gz"
!mkdir -p koten && curl -fsL {ocr} | tar xz -C koten --strip-components 1

In [ ]:
import xml.etree.ElementTree as ET

!cd koten/src && uv run -q --with-requirements ../requirements.txt python ocr.py --sourceimg ../../page.jpg --output ../.. --viz True

for line in ET.parse("page.xml").iter("LINE"):
    print(*(line.get(name) for name in ("TYPE", "X", "Y", "WIDTH", "HEIGHT")), line.get("STRING", "")[:10])
Image.open("viz_page.jpg")

## 3. 領域を囲んで分類する

`regions`の各行は（分類, 左, 上, 幅, 高さ）です。右下の座標は（左＋幅, 上＋高さ）になります。むかでの絵と、和歌一首の領域を`regions`に加えてください。和歌に作者名を含めるかどうかも決めてください。

In [ ]:
regions = [
    ("見出し", 2173, 534, 55, 203),
    ("絵", 750, 1340, 350, 240),
]
partner = [
    ("絵", 560, 1060, 560, 560),
]

marked = Image.open("page.jpg").convert("RGB")
draw = ImageDraw.Draw(marked)
for name, color, rows in (("自分", "blue", regions), ("相手", "orange", partner)):
    for number, (label, x, y, w, h) in enumerate(rows):
        draw.rectangle((x, y, x + w, y + h), outline=color, width=5)
        draw.text((x, y), str(number), fill=color, font_size=40)
        print(name, number, label, (x, y), (x + w, y + h))
marked

## 4. 相手と比べる

3.の`partner`には、境界の違う例が一つ入っています。相手の`regions`の行に置き換えて実行し直すと、相手の枠が橙色で重なります。境界や分類が違う箇所から分類の規則を一文にし、別の領域でも試してください。

## 5. 別の資料で試す

IIIFに対応した資料なら、1.の`url`をその資料の画像URLに変えると、同じ手順で領域を囲めます。

## 確認問題

むかでの絵を囲む領域を、分類「絵」として`regions`に加えてください。3.のセルを実行し直してから、次のセルで答えを確かめてください。

In [ ]:
#@title 答えを確かめる
target = (300, 1430, 130, 260)
widest = 450, 600
missing = "むかでの体を囲む「絵」の領域がありません"
boxes = [(x, y, w, h) for label, x, y, w, h in regions if label == "絵"]
covering = [box for box in boxes if box[0] <= target[0] and box[1] <= target[1]
            and target[0] + target[2] <= box[0] + box[2] and target[1] + target[3] <= box[1] + box[3]]
if not covering:
    print(f"結果：不合格（{missing}）")
elif all(w > widest[0] or h > widest[1] for x, y, w, h in covering):
    print("結果：不合格（領域が広すぎます。対象だけを囲んでください）")
else:
    print("結果：合格")